In [1]:
# Install packages needed for data collection and analysis
%pip install pandas requests python-dotenv

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.7 MB 33.8 MB/s eta 0:00:01
   ------ --------------------------------- 1.6/9.7 MB 3.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.7 MB 3.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.7 MB 2.3 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.7 MB 2.6 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/9.7 MB 2.2 MB/s eta 0:00:04
   ------------- -------------------------- 3.1/9.7 MB 2.2 MB/s eta 0:00:04
   -------------- ------------------------- 3.4/9.7 MB 1.9 MB/s eta 0:00


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Load the OpenAQ API key and set up the API connection
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

API_KEY = os.getenv("OPENAQ_API_KEY")
BASE_URL = "https://api.openaq.org/v3"

headers = {
    "X-API-Key": API_KEY
}

print("API key loaded:", API_KEY is not None)

API key loaded: True


In [2]:
# Test the OpenAQ API connection with a New Delhi monitoring location
url = f"{BASE_URL}/locations/8118"

response = requests.get(url, headers=headers, timeout=30)

print("Status code:", response.status_code)

if response.ok:
    print("OpenAQ API connection successful.")
else:
    print(response.text)

Status code: 200
OpenAQ API connection successful.


In [3]:
# Find air-quality monitoring locations within 25 km of central Delhi
params = {
    "coordinates": "28.6139,77.2090",
    "radius": 25000,
    "limit": 100
}

response = requests.get(
    f"{BASE_URL}/locations",
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()
location_data = response.json()

print("Locations found:", len(location_data["results"]))

Locations found: 96


In [5]:
# Create a summary of the available Delhi monitoring locations
stations = []

for location in location_data["results"]:
    pollutants = sorted({
        sensor["parameter"]["name"]
        for sensor in location.get("sensors", [])
    })

    first_dt = location.get("datetimeFirst") or {}
    last_dt = location.get("datetimeLast") or {}
    coordinates = location.get("coordinates") or {}

    stations.append({
        "location_id": location["id"],
        "station_name": location["name"],
        "latitude": coordinates.get("latitude"),
        "longitude": coordinates.get("longitude"),
        "pollutants": ", ".join(pollutants),
        "first_measurement": first_dt.get("local"),
        "last_measurement": last_dt.get("local")
    })

stations_df = pd.DataFrame(stations)

stations_df

,location_id,station_name,latitude,longitude,pollutants,first_measurement,last_measurement
0,13,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000,"no2, o3, pm25",2016-11-03T00:30:00+05:30,2018-02-22T09:30:00+05:30
1,15,IGI Airport,28.560000,77.094000,"co, no2, o3, pm10, pm25",NaN,NaN
2,16,Civil Lines,28.678700,77.226200,"co, no2, pm10, pm25",NaN,NaN
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05T20:25:00+05:30,2026-09-24T23:00:00+05:30
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05T20:25:00+05:30,2026-09-24T23:00:00+05:30
...,...,...,...,...,...,...,...
91,6254665,"Commonwealth Sports Complex, Delhi - DPCC",28.615828,77.271992,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2026-02-27T22:30:00+05:30,2026-09-24T23:00:00+05:30
92,6254666,"IGNOU_Maidan Garhi, Delhi - DPCC",28.493624,77.201159,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2026-02-27T22:15:00+05:30,2026-09-24T23:00:00+05:30
93,6257818,"Cantonment Area, Delhi - DPCC",28.594169,77.125100,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2026-03-02T18:30:00+05:30,2026-09-24T23:00:00+05:30
94,6299494,"Prashant Garden, Khora - UPPCB",28.611190,77.342060,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2026-04-07T18:15:00+05:30,2026-09-24T22:15:00+05:30


In [6]:
# Filter stations that provide PM2.5 measurements
pm25_stations = stations_df[
    stations_df["pollutants"].str.contains("pm25", case=False, na=False)
].copy()

pm25_stations["first_measurement"] = pd.to_datetime(
    pm25_stations["first_measurement"], errors="coerce", utc=True
)

pm25_stations["last_measurement"] = pd.to_datetime(
    pm25_stations["last_measurement"], errors="coerce", utc=True
)

print("Stations with PM2.5:", len(pm25_stations))

pm25_stations.head()

Stations with PM2.5: 94


,location_id,station_name,latitude,longitude,pollutants,first_measurement,last_measurement
0,13,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000,"no2, o3, pm25",2016-11-02 19:00:00+00:00,2018-02-22 04:00:00+00:00
1,15,IGI Airport,28.560000,77.094000,"co, no2, o3, pm10, pm25",NaT,NaT
2,16,Civil Lines,28.678700,77.226200,"co, no2, pm10, pm25",NaT,NaT
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05 14:55:00+00:00,2026-09-24 17:30:00+00:00
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,"co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05 14:55:00+00:00,2026-09-24 17:30:00+00:00


In [7]:
# Rank PM2.5 stations by the length of their available measurement history
pm25_stations["history_years"] = (
    pm25_stations["last_measurement"] -
    pm25_stations["first_measurement"]
).dt.days / 365.25

station_coverage = (
    pm25_stations[
        [
            "location_id",
            "station_name",
            "pollutants",
            "first_measurement",
            "last_measurement",
            "history_years"
        ]
    ]
    .dropna(subset=["history_years"])
    .sort_values("history_years", ascending=False)
    .reset_index(drop=True)
)

station_coverage.head(15)

,location_id,station_name,pollutants,first_measurement,last_measurement,history_years
0,17,"R K Puram, Delhi - DPCC","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05 14:55:00+00:00,2026-09-24 17:30:00+00:00,10.633812
1,50,"Punjabi Bagh, Delhi - DPCC","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05 14:55:00+00:00,2026-09-24 17:30:00+00:00,10.633812
2,235,"Anand Vihar, New Delhi - DPCC","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2016-02-05 14:15:00+00:00,2026-09-24 17:30:00+00:00,10.633812
3,8118,New Delhi,pm25,2016-11-09 19:00:00+00:00,2026-09-24 10:30:00+00:00,9.869952
4,5404,"Pusa, Delhi - IMD","co, no, no2, nox, o3, pm10, pm25",2017-09-15 06:00:00+00:00,2026-09-24 17:30:00+00:00,9.023956
5,5626,"DTU, New Delhi - CPCB","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2018-03-09 05:30:00+00:00,2026-09-24 17:30:00+00:00,8.544832
6,5630,"Shadipur, Delhi - CPCB","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2018-03-09 06:00:00+00:00,2026-09-24 17:30:00+00:00,8.544832
7,5622,"NSIT Dwarka, Delhi - CPCB","co, no, no2, nox, o3, pm10, pm25, relativehumi...",2018-03-09 06:00:00+00:00,2026-09-24 14:15:00+00:00,8.544832
8,5613,"ITO, New Delhi - CPCB","co, no, no2, nox, o3, pm10, pm25, so2",2018-03-09 05:30:00+00:00,2026-09-24 17:30:00+00:00,8.544832
9,5610,"North Campus, DU, Delhi - IMD","co, no, no2, nox, o3, pm10, pm25",2018-03-09 05:30:00+00:00,2026-09-24 17:30:00+00:00,8.544832


In [8]:
# Get PM2.5 sensor details for selected long-running Delhi stations
candidate_ids = [17, 50, 235]

sensor_records = []

for location_id in candidate_ids:
    response = requests.get(
        f"{BASE_URL}/locations/{location_id}",
        headers=headers,
        timeout=30
    )
    response.raise_for_status()

    location = response.json()["results"][0]

    for sensor in location.get("sensors", []):
        if sensor["parameter"]["name"].lower() == "pm25":
            sensor_records.append({
                "location_id": location_id,
                "station_name": location["name"],
                "sensor_id": sensor["id"],
                "parameter": sensor["parameter"]["name"],
                "units": sensor["parameter"]["units"]
            })

pm25_sensors_df = pd.DataFrame(sensor_records)

pm25_sensors_df

,location_id,station_name,sensor_id,parameter,units
0,17,"R K Puram, Delhi - DPCC",12234787,pm25,µg/m³
1,17,"R K Puram, Delhi - DPCC",35,pm25,µg/m³
2,50,"Punjabi Bagh, Delhi - DPCC",396,pm25,µg/m³
3,50,"Punjabi Bagh, Delhi - DPCC",12234796,pm25,µg/m³
4,235,"Anand Vihar, New Delhi - DPCC",384,pm25,µg/m³
5,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,µg/m³


In [9]:
# Check PM2.5 sensor measurement coverage
coverage_records = []

for _, row in pm25_sensors_df.iterrows():
    response = requests.get(
        f"{BASE_URL}/sensors/{row['sensor_id']}",
        headers=headers,
        timeout=30
    )
    response.raise_for_status()

    sensor = response.json()["results"][0]

    first_dt = sensor.get("datetimeFirst") or {}
    last_dt = sensor.get("datetimeLast") or {}

    coverage_records.append({
        "station_name": row["station_name"],
        "sensor_id": row["sensor_id"],
        "first_measurement": first_dt.get("local"),
        "last_measurement": last_dt.get("local")
    })

sensor_coverage_df = pd.DataFrame(coverage_records)

sensor_coverage_df

,station_name,sensor_id,first_measurement,last_measurement
0,"R K Puram, Delhi - DPCC",12234787,2025-02-19T01:45:00+05:30,2026-09-24T23:00:00+05:30
1,"R K Puram, Delhi - DPCC",35,2016-02-05T20:25:00+05:30,2018-02-22T02:45:00+05:30
2,"Punjabi Bagh, Delhi - DPCC",396,2016-02-05T20:25:00+05:30,2018-02-22T02:45:00+05:30
3,"Punjabi Bagh, Delhi - DPCC",12234796,2025-02-19T01:45:00+05:30,2026-09-24T23:00:00+05:30
4,"Anand Vihar, New Delhi - DPCC",384,2016-02-05T19:45:00+05:30,2018-02-22T02:45:00+05:30
5,"Anand Vihar, New Delhi - DPCC",12235610,2025-02-19T03:45:00+05:30,2026-09-24T23:00:00+05:30


In [11]:
# Find PM2.5 sensors and their actual measurement coverage
sensor_candidates = []

for location in location_data["results"]:
    for sensor in location.get("sensors", []):
        if sensor["parameter"]["name"].lower() != "pm25":
            continue

        response = requests.get(
            f"{BASE_URL}/sensors/{sensor['id']}",
            headers=headers,
            timeout=30
        )
        response.raise_for_status()

        sensor_info = response.json()["results"][0]

        first_dt = sensor_info.get("datetimeFirst") or {}
        last_dt = sensor_info.get("datetimeLast") or {}

        sensor_candidates.append({
            "station_name": location["name"],
            "location_id": location["id"],
            "sensor_id": sensor["id"],
            "first_measurement": first_dt.get("local"),
            "last_measurement": last_dt.get("local")
        })

best_sensors = pd.DataFrame(sensor_candidates)

best_sensors["first_measurement"] = pd.to_datetime(
    best_sensors["first_measurement"], errors="coerce", utc=True
)

best_sensors["last_measurement"] = pd.to_datetime(
    best_sensors["last_measurement"], errors="coerce", utc=True
)

best_sensors["history_years"] = (
    best_sensors["last_measurement"] -
    best_sensors["first_measurement"]
).dt.days / 365.25

best_sensors = (
    best_sensors
    .dropna(subset=["first_measurement", "last_measurement"])
    .sort_values("history_years", ascending=False)
    .reset_index(drop=True)
)

best_sensors.head(20)

,station_name,location_id,sensor_id,first_measurement,last_measurement,history_years
0,New Delhi,8118,23534,2016-11-09 19:00:00+00:00,2026-09-24 10:30:00+00:00,9.869952
1,"NSIT Dwarka, Delhi - CPCB",5622,14922,2018-03-09 06:00:00+00:00,2022-10-31 02:00:00+00:00,4.643395
2,"CRRI Mathura Road, New Delhi - IMD",5627,14935,2018-03-09 05:30:00+00:00,2022-10-31 01:15:00+00:00,4.643395
3,"Anand Vihar, Delhi - DPCC",5509,14521,2018-03-09 03:30:00+00:00,2022-10-31 00:30:00+00:00,4.643395
4,"Sector- 16A, Faridabad - HSPCB",5617,14930,2018-03-09 06:00:00+00:00,2022-10-31 02:00:00+00:00,4.643395
5,"Shadipur, Delhi - CPCB",5630,14959,2018-03-09 06:00:00+00:00,2022-10-31 02:00:00+00:00,4.643395
6,"North Campus, DU, Delhi - IMD",5610,14860,2018-03-09 06:00:00+00:00,2022-10-31 02:00:00+00:00,4.643395
7,"Vasundhara, Ghaziabad - UPPCB",5665,15239,2018-03-09 06:00:00+00:00,2022-10-31 02:00:00+00:00,4.643395
8,"DTU, New Delhi - CPCB",5626,14985,2018-03-09 05:30:00+00:00,2022-10-31 01:45:00+00:00,4.643395
9,"Lodhi Road, New Delhi - IMD",5634,14984,2018-03-09 05:30:00+00:00,2022-10-16 10:15:00+00:00,4.605065


In [12]:
# Check actual daily PM2.5 data availability for the strongest sensor
sensor_id = 23534

params = {
    "date_from": "2016-11-09",
    "date_to": "2026-09-24",
    "limit": 1000
}

response = requests.get(
    f"{BASE_URL}/sensors/{sensor_id}/days",
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()
daily_data = response.json()

print("Records returned:", len(daily_data["results"]))
print("Total records available:", daily_data["meta"]["found"])

Records returned: 1000
Total records available: >1000


In [13]:
# Download all available daily PM2.5 measurements using pagination
sensor_id = 23534
all_records = []
page = 1

while True:
    params = {
        "date_from": "2016-11-09",
        "date_to": "2026-09-24",
        "limit": 1000,
        "page": page
    }

    response = requests.get(
        f"{BASE_URL}/sensors/{sensor_id}/days",
        headers=headers,
        params=params,
        timeout=30
    )
    response.raise_for_status()

    records = response.json()["results"]

    if not records:
        break

    all_records.extend(records)
    print(f"Page {page}: {len(records)} records")

    if len(records) < 1000:
        break

    page += 1

print("\nTotal records downloaded:", len(all_records))

Page 1: 1000 records
Page 2: 1000 records
Page 3: 1000 records
Page 4: 495 records

Total records downloaded: 3495


In [14]:
# Convert downloaded PM2.5 records into a simple DataFrame
air_quality_df = pd.DataFrame([
    {
        "date": record["period"]["datetimeFrom"]["local"],
        "pm25": record["value"]
    }
    for record in all_records
])

air_quality_df["date"] = pd.to_datetime(
    air_quality_df["date"],
    errors="coerce"
)

air_quality_df = (
    air_quality_df
    .sort_values("date")
    .reset_index(drop=True)
)

air_quality_df.head()

,date,pm25
0,2016-11-09 00:00:00+05:30,270.0
1,2016-11-10 00:00:00+05:30,259.0
2,2016-11-11 00:00:00+05:30,274.0
3,2016-11-12 00:00:00+05:30,238.0
4,2016-11-13 00:00:00+05:30,175.0


In [15]:
# Check dataset quality and historical coverage
print("Shape:", air_quality_df.shape)
print("Date range:", air_quality_df["date"].min(), "to", air_quality_df["date"].max())
print("Missing values:\n", air_quality_df.isna().sum())
print("Duplicate dates:", air_quality_df["date"].duplicated().sum())

expected_days = pd.date_range(
    air_quality_df["date"].min().normalize(),
    air_quality_df["date"].max().normalize(),
    freq="D"
)

actual_days = pd.DatetimeIndex(
    air_quality_df["date"].dt.normalize().unique()
)

missing_days = expected_days.difference(actual_days)

print("Expected days:", len(expected_days))
print("Available days:", len(actual_days))
print("Missing days:", len(missing_days))
print("Data coverage:", f"{len(actual_days) / len(expected_days) * 100:.2f}%")

Shape: (3495, 2)
Date range: 2016-11-09 00:00:00+05:30 to 2026-09-24 00:00:00+05:30
Missing values:
 date     0
pm25    92
dtype: int64
Duplicate dates: 0
Expected days: 3607
Available days: 3495
Missing days: 112
Data coverage: 96.89%


In [16]:
# Save the raw historical PM2.5 dataset for the cleaning stage
output_path = project_root / "data" / "raw" / "air_quality.csv"

air_quality_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(air_quality_df))

Saved: c:\Users\aayus\OneDrive\Desktop\Data Science Project\Air Quality Trend & Forecast\data\raw\air_quality.csv
Rows: 3495
